> ⏱ ~15 min

# Scenario

**Section 1.** The travel desk hires three specialists — flight, hotel, and car — and one concierge who routes each request to the right specialist. In this notebook you assemble all four with Microsoft Agent Framework (MAF) and prove the concierge can plan a multi-leg trip end to end.

The pattern we use is called **agents as tools**. The concierge does not know about catalogs or booking functions. It only knows it can call `flight_agent`, `hotel_agent`, and `car_agent`. Each specialist owns its own catalog tools and decides how to use them.


## 2. What you will do

1. Create one shared `FoundryChatClient` for all four agents.
2. Build the three specialist agents with the the workshop's booking tools attached.
3. Build the concierge orchestrator that exposes each specialist as a single tool.
4. Run a multi-leg request and inspect the response.

All four agents share one model deployment because Cohere `command-a` is fast enough to play every role in this small example. In production you might use different deployments for different specialists.

---


In [ ]:
# Suppress experimental/deprecation warnings to keep the learning output clean.
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os, sys
from dotenv import load_dotenv

# Locate this lab folder portably (do not hardcode the repo name; learners may fork+rename).
LAB_DIR = Path.cwd().resolve()
if LAB_DIR.name != 'lab-1-foundry-maf':
    for _p in (LAB_DIR, *LAB_DIR.parents):
        _candidate = _p / 'cohere' / 'lab-1-foundry-maf'
        if _candidate.exists():
            LAB_DIR = _candidate
            break
COHERE_DIR = LAB_DIR.parent
load_dotenv(COHERE_DIR / '.env')
sys.path.insert(0, str(LAB_DIR))

from agents import (
    make_chat_client,
    build_flight_agent,
    build_hotel_agent,
    build_car_agent,
    build_concierge,
)

client = make_chat_client()
print('Chat client ready:', type(client).__name__)


## 3. Build the three specialists

Each specialist agent has its own instructions and the matching the workshop's booking tools. The factory functions live in `agents/travel_agents.py` so the same builders are reusable from later notebooks (evaluation and red-team).


In [ ]:
flight_agent = build_flight_agent(client)
hotel_agent  = build_hotel_agent(client)
car_agent    = build_car_agent(client)
for agent in (flight_agent, hotel_agent, car_agent):
    print(f'{agent.name:<14}→ {agent.description}')


## 4. Run a specialist on its own

Before adding the concierge, send a flight-only request straight to `flight_agent`. This proves the booking tools and the catalog data are visible from the local agent runtime.


In [ ]:
response = await flight_agent.run(
    'Find an economy flight from SFO to NYC on 2026-07-10 returning 2026-07-13 under $400.'
)
print(response.text)


## 5. Assemble the concierge orchestrator

The concierge sees three tools (`flight_agent`, `hotel_agent`, `car_agent`). Each tool forwards a natural-language request to the matching specialist and returns the specialist's answer as plain text. The concierge then composes those answers into a single itinerary.

Because the specialists are also `Agent` instances under the hood, every booking decision still flows through the same Cohere `command-a` deployment.


In [ ]:
concierge = build_concierge(client)
concierge_tools = concierge.default_options.get('tools', [])
print('Concierge tools:', [t.name for t in concierge_tools])


## 6. Run a multi-leg trip end to end

This request needs all three specialists. The concierge should call `flight_agent`, `hotel_agent`, and `car_agent` in turn and combine their results into a compliant Chicago itinerary.


In [ ]:
response = await concierge.run(
    'Find a flight, hotel, and standard car for Chicago from 2026-07-18 to 2026-07-21. '
    'Departure city is SFO. Keep everything within corporate policy.'
)
print(response.text)


In [ ]:
response = await concierge.run(
    "I need to plan a 2-leg business trip in July 2026 with flight + hotel for each stop, "
    "no rental cars (we'll use public transit / walk everywhere). "
    "Leg 1: San Francisco (SFO) to Seattle (SEA) on 2026-07-11, economy under $250; "
    "one night in downtown Seattle, max $275/night. "
    "Leg 2: Seattle (SEA) to Chicago (CHI) on 2026-07-12, economy under $350; "
    "stay in Chicago through 2026-07-18, max $230/night. "
    "Confirm one flight and one hotel for each leg, no car bookings on any leg, "
    "and give me a single itinerary with total estimated cost."
)
print(response.text)


## 7. What you confirmed

1. MAF builds Cohere-powered specialist agents from a single shared chat client.
2. The concierge can orchestrate the three specialists through the agents-as-tools pattern.
3. The workshop's booking tools and catalog files work unchanged inside the local agent runtime.

Next, `03-trace-multi-agent.ipynb` turns on MAF's OpenTelemetry tracing and re-runs the concierge so you can see every specialist call, tool invocation, and latency hop — locally and (optionally) in the Foundry Monitoring tab.
